### Bước 1: Kiểm tra GPU


In [ ]:
!nvidia-smi

### Bước 2: Cài đặt `uv` và Python 3.12

In [ ]:
!pip install -q uv
!uv python install 3.12

### Bước 3: Tải mã nguồn dự án


In [ ]:
REPO_URL = "https://github.com/luatducvo/3D-scene.git"

import os
if not os.path.exists("/content/3D-scene"):
    !git clone $REPO_URL /content/3D-scene

%cd /content/3D-scene

### Bước 4: Cài đặt thư viện phụ thuộc cho `scene-compute`
- Chạy với `--extra gpu` để cài đặt đầy đủ Depth Anything 3, SAM 3, GeoCalib, MetaCLIP 2.

In [ ]:
# Cài đặt GPU packages
!uv sync --python 3.12 --package scene-compute --extra gpu

### Bước 5: Tải trước model weights






In [ ]:
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

import os

DRIVE_DIR = "/content/drive/MyDrive/model-weights"
MODELS_DIR = f"{DRIVE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Thiết lập biến môi trường trỏ vào Google Drive
os.environ["DATA_DIR"] = "/content/data"        # Data scene tạm để ở SSD Colab cho nhanh
os.environ["MODELS_DIR"] = MODELS_DIR          # Weights lưu vĩnh viễn trên Drive
os.environ["HF_HOME"] = f"{MODELS_DIR}/hf"
os.environ["TORCH_HOME"] = f"{MODELS_DIR}/torch"
os.environ["MODELS_PIPELINE_CONFIG"] = "/content/3D-scene/backend/config/pipeline.yaml"

# Nếu bạn dùng SAM 3 gated weights từ HuggingFace, bỏ comment dòng dưới:
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

!uv run --python 3.12 python -m scene_compute.download_models

### Bước 6: Khởi chạy Cloudflare Tunnel
Tạo đường hầm HTTPS an toàn để kết nối từ máy tính của bạn về Colab.

In [ ]:
import subprocess, time, re

# Cài đặt cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Chạy tunnel nền chuyển tiếp port 9000
!pkill cloudflared || true
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:9001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("URL PUBLIC CHO MÁY LOCAL:", public_url)

### Bước 7: Khởi chạy `scene-compute` HTTP Server


In [ ]:
import os
from google.colab import userdata

DRIVE_DIR = "/content/drive/MyDrive/model-weights"
MODELS_DIR = f"{DRIVE_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)

COMPUTE_TOKEN = userdata.get('COMPUTE_TOKEN')

os.environ["COMPUTE_TOKEN"] = COMPUTE_TOKEN
os.environ["PORT"] = "9000"
os.environ["DATA_DIR"] = "/content/data"
os.environ["MODELS_DIR"] = MODELS_DIR
os.environ["HF_HOME"] = f"{MODELS_DIR}/hf"
os.environ["TORCH_HOME"] = f"{MODELS_DIR}/torch"
os.environ["CHUNK_SIZE_MB"] = "25"

print(f"Starting scene-compute on port 9000 with token: {COMPUTE_TOKEN}")
!uv run --python 3.12 uvicorn scene_compute.server:app --host 0.0.0.0 --port 9001